# PCI DSS assistant

## Step 1 — PDF into a list of documents

In [1]:
import ingest

In [2]:
# downloads the PCI DSS v4.0.1 PDF into data/ (~4.4 MB, skipped if already there)
ingest.download_pdf()

'data/pci-dss-v4_0_1.pdf'

In [3]:
documents = ingest.load_documents()
len(documents)

261

In [4]:
# what one document looks like
doc = documents[14]

doc['page'], doc['req_ids'], doc['requirement']

(59, '1.4.2', '1')

In [5]:
print(doc['text'])

Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 55 
 
Requirements and Testing Procedures 
Guidance 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Ensuring that public access to a system 
component is specifically authorized reduces the 
risk of system components being unnecessarily 
exposed to untrusted networks. 
Good Practice 
System components that provide publicly 
accessible services, such as email, web, and 
DNS servers, are the most vulnerable to threats 
originating from untrusted networks.  
Ideally, such systems are placed within a 
dedicated trusted network that is public facing (for 
example, a DMZ) but that is separated via NSCs 
from more sensitive internal systems, which helps 
protect the rest of the network in the event these 
externally accessible systems are compromised. 
This functionality is intended to p

In [6]:
# how long are the documents?
lengths = [len(d['text']) for d in documents]

min(lengths), sum(lengths) // len(lengths), max(lengths)

(791, 2153, 3623)

## Step 2 — Text search

`minsearch` builds a TF-IDF index over the text fields and lets us filter on the
keyword fields. No server, no persistence: 261 documents are indexed in a moment.

In [7]:
index = ingest.build_index(documents)


def show(results):
    for r in results:
        print(f"page {r['page']:>3}  req {r['req_ids']}")

In [8]:
query = 'how long must audit logs be retained'
results = index.search(query, num_results=5)

show(results)

page 256  req 10.5, 10.5.1
page 134  req 5.3.4
page 243  req 10.2, 10.2.1, 10.2.1.1
page 244  req 10.2.1.2, 10.2.1.3, 10.2.1.4
page 248  req 10.3, 10.3.1


In [9]:
# the top hit, to check it really answers the question
print(results[0]['text'][:800])

Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 252 
 
Requirements and Testing Procedures 
Guidance 
10.5 Audit log history is retained and available for analysis. 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Retaining historical audit logs for at least 12 
months is necessary because compromises often 
go unnoticed for significant lengths of time. 
Having centrally stored log history allows 
investigators to better determine the length of 
time a potential breach was occurring, and the 
possible system(s) impacted. By having three 
months of logs immediately available, an entity 
can quickly identify and minimize impact of a d


In [10]:
show(index.search('do we need MFA for remote access', num_results=5))

page 208  req 8.5, 8.5.1
page 206  req 8.4.3
page 202  req 8.4, 8.4.1
page 203  req 8.4.2
page 184  req 8.2.2, 8.2.3


### Requirement numbers, and why they nearly did not work

Half of the realistic questions about a standard are about a specific requirement
number. That turned out to be broken by default.

In [11]:
from minsearch import Index

# an index built exactly the way the course did it, with no extra parameters
plain = Index(text_fields=['text', 'req_ids'], keyword_fields=['requirement'])
plain.fit(documents)

plain.search('8.3.6', num_results=3)  # -> []

[]

Nothing. `minsearch` uses scikit-learn's `TfidfVectorizer`, whose default token
pattern is `\b\w\w+\b`: a token needs at least two word characters, and a dot is
not a word character. So `8.3.6` is split into `8`, `3`, `6`, each too short to
survive — requirement numbers never make it into the index.

Letting a dot stay inside a token fixes it. That is the one extra line in
`build_index`:

```python
vectorizer_params = {'token_pattern': r'(?u)\b\w[\w.]*\b'}
```

In [12]:
show(index.search('8.3.6', num_results=3))

page 194  req 8.3.6
page  69  req 2.2.2


In [13]:
# it helps ordinary questions too — 10.5.1 is the requirement about log retention
show(index.search('10.5.1', num_results=3))

page 256  req 10.5, 10.5.1
page 134  req 5.3.4


### Boosting and filtering

`boost_dict` weights the text fields against each other, `filter_dict` does exact
matching on a keyword field — the role `course` played in the course code.

In [14]:
# make an exact requirement number outweigh pages that merely mention it
show(index.search('8.3.6', num_results=3, boost_dict={'req_ids': 3.0, 'text': 1.0}))

page 194  req 8.3.6
page  69  req 2.2.2


In [15]:
# search only inside Requirement 3 (Protect Stored Account Data)
show(index.search(
    'can we store the card verification code',
    num_results=5,
    filter_dict={'requirement': '3'}
))

page  85  req 3.3.1.2
page  89  req 3.3.3
page  83  req 3.3, 3.3.1
page 113  req 3.7.9
page  84  req 3.3.1.1


Good enough to build on. Which boost values are actually best, and whether TF-IDF
beats embeddings here, is not something to decide by eye — that is measured in
step 5.

## Step 3 — First RAG

Search finds the pages; the LLM turns them into an answer. Three stages:
**R**etrieval, **A**ugmentation (putting what we found into the prompt),
**G**eneration.

In [16]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [17]:
from rag_helper import RAGBase

pci_rag = RAGBase(index=index, llm_client=client)

### Look at the pieces before running the whole thing

In [18]:
query = 'how long must we retain audit logs?'

search_results = pci_rag.search(query)
show(search_results)

page 244  req 10.2.1.2, 10.2.1.3, 10.2.1.4
page 243  req 10.2, 10.2.1, 10.2.1.1
page 256  req 10.5, 10.5.1
page 248  req 10.3, 10.3.1
page 246  req 10.2.1.5, 10.2.1.6


In [19]:
# what the model will actually see
prompt = pci_rag.build_prompt(query, search_results)

print(prompt[:1500])

QUESTION: how long must we retain audit logs?

CONTEXT:
[requirement 10.2.1.2, 10.2.1.3, 10.2.1.4, page 244]
Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 240 
 
Requirements and Testing Procedures 
Guidance 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Accounts with increased access privileges, such 
as the “administrator” or “root” account, have the 
potential to significantly impact the security or 
operational functionality of a system. Without a 
log of the activities performed, an organization is 
cannot trace any issues resulting from an 
administrative mistake or misuse of privilege back 
to the specific action and account. 
Definitions 
The functions or activities considered to be 
administrative are beyond those performed by 
regular users as part of routine business 
functions. 
Refer to Appendix G for the defini

In [20]:
print(f'prompt length: {len(prompt)} characters')

prompt length: 10039 characters


### The whole pipeline

In [26]:
answer = pci_rag.rag(query)

print(answer)

Audit log history must be retained for at least 12 months, with at least the most recent three months immediately available for analysis. (req. 10.5.1, p. 252)


In [22]:
print(pci_rag.rag('do we need MFA for administrative access, or only for remote access?'))

For administrative access to the CDE, MFA is required for all **non-console** access, not just remote access. (req. 8.4.1, p. 202)

For remote access originating from outside the entity’s network that could access or impact the CDE, MFA is also required. (req. 8.4.3, p. 202)

So the answer is: **both**—MFA is required for non-console administrative access to the CDE, and separately for remote access from outside the entity’s network that could access or impact the CDE. (req. 8.4.1, p. 202; req. 8.4.3, p. 202)


In [23]:
print(pci_rag.rag('can we store the CVV after the transaction is authorized?'))

No. The card verification code (CVV) is sensitive authentication data and is not allowed to be stored upon completion of the authorization process, even if encrypted. It must be rendered unrecoverable after authorization. (req. 3.3.1.2, p. 85; req. 3.3.1, p. 83)


In [24]:
print(pci_rag.rag('what exactly does requirement 8.3.6 ask for?'))

Requirement 8.3.6 says that if passwords/passphrases are used as an authentication factor to meet Requirement 8.3.1, they must meet these minimum complexity rules: at least 12 characters long, or 8 characters if the system does not support 12, and they must contain both numeric and alphabetic characters. (req. 8.3.6, p. 194)

It also says to examine system configuration settings to verify the password/passphrase complexity parameters are set to meet all elements of the requirement. (req. 8.3.6, p. 194)

A few scope notes are included: it does not apply to user accounts on POS terminals that access only one card number at a time for a single transaction, and it does not apply to application or system accounts governed by section 8.6. (req. 8.3.6, p. 194) It is also marked as a best practice until 31 March 2025, after which it becomes required. (req. 8.3.6, p. 194)


### Does it refuse when it should?

A compliance assistant that confidently answers questions the standard does not
cover is worse than one that stays silent. This question has nothing to do with
PCI DSS, so the answer should be "I don't know".

In [25]:
print(pci_rag.rag('what is the maximum fine under GDPR?'))

I don't know.


Working end to end. Two things we still cannot say anything about:

- how often search puts the right page in the top 5 — measured in step 5;
- whether this prompt is better than another one — measured in step 7.

Both need a set of questions with known answers, which is step 4.